<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/02-data-structures/04_mini_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Mini Project: A Naive Bayes Spam Filter from Scratch

In this project you'll build a spam filter that learns from example messages. It uses a classic algorithm called **Naive Bayes**, which was one of the first successful spam filtering methods and is still a strong baseline for text classification today.

What makes it a good project for this module is that the whole model is just a handful of counters. Training means counting words. Predicting means looking up those counts. Every step uses the structures from the lesson: `Counter`, `defaultdict`, sets, dicts and `heapq`.

By the end you'll have:

1. a small labelled dataset of messages
2. a tokenizer
3. a stratified train/test split
4. a Naive Bayes model, trained by counting
5. predictions in log space, and an explanation of why that's necessary
6. an evaluation with accuracy, precision and recall
7. a look inside the model at the words it considers most "spammy"

## Step 1: The dataset

Here are 70 short messages, written for this project: 30 spam and 40 normal messages (usually called "ham"). Some ham messages deliberately use words like "free", "won", "prize" and "call", so the model can't just flag any message containing those words.

A real dataset would be much bigger. The well-known SMS Spam Collection, for example, has about 5,500 messages. Module 5 covers loading real data from files. The code you write here works the same way on a bigger dataset.

In [1]:
spam = [
    "Congratulations! You have won a free iPhone. Click here to claim your prize now",
    "URGENT: your account has been suspended. Verify your details at the link below",
    "Win cash prizes every week! Text WIN to 80085 to enter",
    "You have been selected for a free holiday to Bali. Call now to claim",
    "Limited offer: get 50% off all watches today only. Buy now",
    "Your loan has been approved! Get up to $5000 cash today, no credit check",
    "Claim your free gift card now before it expires tonight",
    "Hot singles in your area are waiting to meet you. Click here",
    "Final notice: you owe unpaid taxes. Pay now to avoid arrest",
    "Earn $500 a day working from home. No experience needed. Reply YES",
    "You are our lucky winner! Reply with your bank details to receive your prize",
    "Free entry into our weekly draw. Text CLAIM to 55555 now",
    "Cheap meds online, no prescription needed. Order today",
    "Your parcel could not be delivered. Click the link to pay the delivery fee",
    "Act now! This exclusive offer ends at midnight. Free shipping on all orders",
    "Dear customer, your password expires today. Click here to keep your account",
    "Get rich fast with this one simple crypto trick. Join now for free",
    "You have won 1 million dollars in our lottery. Send your details to claim",
    "Lose 10 kg in 10 days with this miracle pill. Order now",
    "Congratulations, you qualify for a free credit card with zero interest",
    "Exclusive deal for you: win a brand new car. Click to enter the draw",
    "Your bank account is locked. Verify your identity now at this link",
    "Double your money in one week guaranteed. Invest today",
    "Claim your cash reward now. Offer valid for 24 hours only",
    "FREE ringtones! Text TONE to 12345 to get yours now",
    "We tried to reach you about your car warranty. Call us back now",
    "Special promotion: buy one get one free on all products today",
    "You have an unclaimed prize waiting. Click here to collect it",
    "Make money online fast. Thousands of people are already earning. Join free",
    "Your subscription will renew for $99. Call now to cancel",
]

ham = [
    "Hey, are we still meeting for lunch tomorrow?",
    "Can you send me the slides from today's meeting?",
    "I'm running ten minutes late, sorry",
    "Happy birthday! Hope you have a great day",
    "Did you finish the assignment for the machine learning class?",
    "Call me when you get home",
    "Are you free this weekend? We could go hiking",
    "The training run finished overnight, accuracy is up to 91 percent",
    "Mom says dinner is at seven",
    "Thanks for your help with the project yesterday",
    "Can you pick up some milk on your way back?",
    "Our team won the match today! Great game everyone",
    "Don't forget the meeting has moved to 3pm",
    "I just pushed the fix to GitHub, can you review it?",
    "Let's grab coffee after class",
    "The library is closed tomorrow, so let's study at my place",
    "Good luck with your exam today, you'll do great",
    "I'll call you back in five minutes",
    "Did you see the new episode last night?",
    "Please review the report before Friday",
    "The bus is late again, I'll be there soon",
    "Can we reschedule our call to next week?",
    "Nice work on the presentation, the client loved it",
    "Are you coming to the party on Saturday?",
    "I left my charger at your place, can I get it tomorrow?",
    "The GPU server is free now if you want to run your experiments",
    "Remember to bring your laptop to the workshop",
    "We won the hackathon! The prize is a trip to the conference",
    "What time does the movie start?",
    "Thanks for dinner, it was lovely",
    "Can you share the dataset link again? The old one expired",
    "Let me know when you are free to talk",
    "I'm at the station now, where are you?",
    "The landlord will fix the heater on Monday",
    "My flight lands at 6, can you pick me up?",
    "Do you want to join our study group for the stats exam?",
    "Your order from the bookstore is ready for pickup",
    "Sorry I missed your call, I was in a meeting",
    "The code review is done, just a few small comments",
    "See you at the gym at 6 tomorrow",
]

dataset = [(text, "spam") for text in spam] + [(text, "ham") for text in ham]
print(len(dataset), "messages")

70 messages


## Step 2: Tokenize

We need to turn each message into a list of words. We'll lowercase everything and treat any character that isn't a letter or digit as a space. So `"FREE ringtones!"` becomes `["free", "ringtones"]` and `"$5000"` becomes `["5000"]`.

In [2]:
def tokenize(text):
    cleaned = "".join(ch if ch.isalnum() else " " for ch in text.lower())
    return cleaned.split()


print(tokenize("Congratulations! You have WON a free iPhone."))
print(tokenize("Get up to $5000 cash today, no credit check"))
print(tokenize("Can you send me the slides from today's meeting?"))

['congratulations', 'you', 'have', 'won', 'a', 'free', 'iphone']
['get', 'up', 'to', '5000', 'cash', 'today', 'no', 'credit', 'check']
['can', 'you', 'send', 'me', 'the', 'slides', 'from', 'today', 's', 'meeting']


`today's` becomes `today` and `s`. That's not perfect, but it's consistent, and for counting words that's what matters most.

## Step 3: Stratified train/test split

We have 30 spam and 40 ham messages. With such a small dataset, a random split could easily put most of the spam in one set. So we shuffle, then use a stratified split (exercise 10) to keep the 30/40 balance in both sets.

In [3]:
import random
from collections import Counter, defaultdict


def stratified_split(samples, test_ratio):
    positions = defaultdict(list)
    for i, (_, label) in enumerate(samples):
        positions[label].append(i)

    test_positions = set()
    for idx in positions.values():
        n_test = round(len(idx) * test_ratio)
        test_positions.update(idx[len(idx) - n_test:])

    train = [s for i, s in enumerate(samples) if i not in test_positions]
    test = [s for i, s in enumerate(samples) if i in test_positions]
    return train, test


shuffled = dataset.copy()
random.Random(3).shuffle(shuffled)
train, test = stratified_split(shuffled, test_ratio=0.3)

print("train:", len(train), Counter(label for _, label in train))
print("test: ", len(test), Counter(label for _, label in test))

train: 49 Counter({'ham': 28, 'spam': 21})
test:  21 Counter({'ham': 12, 'spam': 9})


## Step 4: How Naive Bayes works

We want to know: given the words in a message, is it more likely to be spam or ham?

Bayes' rule lets us flip that question around into things we can count:

$$P(\text{spam} \mid \text{words}) \propto P(\text{spam}) \times P(\text{words} \mid \text{spam})$$

- **P(spam)** is the *prior*: what fraction of all messages are spam. Easy, just count.
- **P(words | spam)** is how likely these words are to appear in a spam message.

The second one is hard to estimate for a whole message, because almost every message is unique. So Naive Bayes makes a big simplifying assumption: it treats each word as independent of the others. Then the probability of the whole message is just the product of the probabilities of each word:

$$P(\text{words} \mid \text{spam}) \approx P(w_1 \mid \text{spam}) \times P(w_2 \mid \text{spam}) \times \dots$$

That's the "naive" part. It's obviously not true ("credit" and "card" are not independent), but it works surprisingly well in practice.

And **P(word | spam)** is just: how many times did this word appear in spam messages, divided by the total number of words in spam messages. More counting.

So training a Naive Bayes model means counting three things:

1. how many messages there are of each class
2. how many times each word appears in each class
3. the set of all words seen (the vocabulary)

## Step 5: Train

Here's the whole training step. It's just counting, with the right data structure for each count.

In [4]:
def train_naive_bayes(samples):
    class_counts = Counter()              # messages per class
    word_counts = defaultdict(Counter)    # word counts per class
    vocab = set()                         # every word seen

    for text, label in samples:
        tokens = tokenize(text)
        class_counts[label] += 1
        word_counts[label].update(tokens)
        vocab.update(tokens)

    return {
        "class_counts": class_counts,
        "word_counts": word_counts,
        "total_words": {label: sum(c.values()) for label, c in word_counts.items()},
        "vocab": vocab,
    }


model = train_naive_bayes(train)

print("classes:", dict(model["class_counts"]))
print("vocabulary size:", len(model["vocab"]))
print("total words per class:", model["total_words"])
print("top spam words:", model["word_counts"]["spam"].most_common(8))
print("top ham words: ", model["word_counts"]["ham"].most_common(8))

classes: {'ham': 28, 'spam': 21}
vocabulary size: 260
total words per class: {'ham': 249, 'spam': 250}
top spam words: [('your', 14), ('to', 14), ('now', 8), ('you', 7), ('free', 6), ('claim', 5), ('for', 5), ('click', 5)]
top ham words:  [('the', 18), ('you', 11), ('i', 8), ('to', 8), ('your', 7), ('can', 6), ('is', 4), ('at', 4)]


The most common words in both classes are boring ones like "you", "the" and "to". That's fine. What matters for classification is how much *more* common a word is in one class than the other, and the model will work that out from the counts.

## Step 6: Two problems to solve before predicting

### Problem 1: words we've never seen in a class

Suppose the word "hackathon" appeared in ham but never in spam. Then P("hackathon" | spam) = 0 / total = 0. And since we multiply all the word probabilities together, one zero makes the whole product zero. A single unseen word would make a message "impossible" as spam, no matter what else it says.

The fix is **Laplace smoothing**: pretend every word in the vocabulary was seen one extra time in every class.

$$P(w \mid c) = \frac{\text{count}(w, c) + 1}{\text{total words in } c + |V|}$$

where |V| is the vocabulary size. Now nothing is ever exactly zero, and rare words don't dominate.

### Problem 2: multiplying lots of small numbers

Word probabilities are small, often below 0.01. Multiply a few hundred of them and you get a number so small that a float can't represent it. It just becomes 0.0. This is called *underflow*:

In [5]:
import math

p = 0.001
product = 1.0
for _ in range(120):
    product *= p
print(product)        # underflows to exactly 0.0

log_sum = sum(math.log(p) for _ in range(120))
print(log_sum)        # no problem at all

0.0
-828.9306334778564


The fix is to work with logarithms. Since `log(a * b) = log(a) + log(b)`, we can **add log probabilities** instead of multiplying probabilities. The class with the biggest log score is still the most likely class, because `log` never changes which of two numbers is bigger.

Every serious ML library does this. When you see "log-likelihood" or "log-softmax" in PyTorch, this is why.

## Step 7: Predict

In [6]:
def log_scores(model, text):
    n_messages = sum(model["class_counts"].values())
    vocab_size = len(model["vocab"])
    tokens = tokenize(text)

    scores = {}
    for label, n in model["class_counts"].items():
        score = math.log(n / n_messages)                       # prior
        counts = model["word_counts"][label]
        denominator = model["total_words"][label] + vocab_size
        for token in tokens:
            if token not in model["vocab"]:
                continue                                       # skip words never seen in training
            score += math.log((counts[token] + 1) / denominator)
        scores[label] = score
    return scores


def predict(model, text):
    scores = log_scores(model, text)
    return max(scores, key=scores.get)


msg = "Claim your free prize now"
print(log_scores(model, msg))
print(predict(model, msg))

{'ham': -27.445189751671982, 'spam': -22.277794803589153}
spam


A few details worth pointing out:

- `counts[token]` works even for words never seen in that class, because `counts` is a `Counter` and missing keys return 0.
- Words that aren't in the vocabulary at all are skipped. We know nothing about them, so they shouldn't push the score either way.
- `token not in model["vocab"]` is a set lookup, so it's O(1). With a list it would scan the whole vocabulary for every word of every message.

The log scores are hard to read, since they're just large negative numbers. To turn them into probabilities that add up to 1, we use a function called softmax. There's one trick: computing `exp(-48.2)` directly is fine, but `exp(-1000)` underflows to 0. Subtracting the biggest score first keeps the numbers in a safe range without changing the result:

In [7]:
def to_probabilities(scores):
    biggest = max(scores.values())
    exps = {label: math.exp(s - biggest) for label, s in scores.items()}
    total = sum(exps.values())
    return {label: e / total for label, e in exps.items()}


for msg in ["Claim your free prize now", "Are we still meeting tomorrow?", "Call me now"]:
    probs = to_probabilities(log_scores(model, msg))
    print(f"{msg!r:35}  spam={probs['spam']:.1%}")

'Claim your free prize now'          spam=99.4%
'Are we still meeting tomorrow?'     spam=4.0%
'Call me now'                        spam=42.7%


## Step 8: Evaluate

Now let's see how it does on the test messages it has never seen.

In [8]:
predictions = [(text, label, predict(model, text)) for text, label in test]

correct = sum(label == guess for _, label, guess in predictions)
print(f"accuracy: {correct}/{len(test)} = {correct / len(test):.1%}")

accuracy: 20/21 = 95.2%


Accuracy alone doesn't tell the whole story. For a spam filter, the two kinds of mistakes aren't equally bad. A spam message landing in your inbox is annoying. A real message from your boss landing in the spam folder could be a serious problem. So we look at precision and recall, which you implemented in Module 1, treating spam as the positive class:

In [9]:
tp = sum(label == "spam" and guess == "spam" for _, label, guess in predictions)
fp = sum(label == "ham" and guess == "spam" for _, label, guess in predictions)
fn = sum(label == "spam" and guess == "ham" for _, label, guess in predictions)

precision = tp / (tp + fp) if tp + fp else 0.0
recall = tp / (tp + fn) if tp + fn else 0.0

print(f"precision: {precision:.1%}  (of messages flagged as spam, how many really were)")
print(f"recall:    {recall:.1%}  (of all spam, how much we caught)")

print("\nmistakes:")
for text, label, guess in predictions:
    if label != guess:
        print(f"  [{label} -> {guess}] {text}")

precision: 100.0%  (of messages flagged as spam, how many really were)
recall:    88.9%  (of all spam, how much we caught)

mistakes:
  [spam -> ham] Earn $500 a day working from home. No experience needed. Reply YES


With only 70 training messages, a few mistakes are expected. Look at the misclassified messages and think about *why* the model got them wrong. Usually it's because the message contains words that only appeared in the other class during training, or because its important words never appeared in training at all.

## Step 9: Look inside the model

One nice thing about Naive Bayes is that you can see exactly what it learned. For each word, compare how likely it is in spam vs ham. The log of that ratio is positive for spammy words and negative for hammy ones.

We only look at words seen at least twice, since a word seen once tells us very little. And we use `heapq.nlargest` and `heapq.nsmallest` to grab the extremes without sorting the whole vocabulary.

In [10]:
import heapq


def word_log_ratio(model, word):
    vocab_size = len(model["vocab"])
    p_spam = (model["word_counts"]["spam"][word] + 1) / (model["total_words"]["spam"] + vocab_size)
    p_ham = (model["word_counts"]["ham"][word] + 1) / (model["total_words"]["ham"] + vocab_size)
    return math.log(p_spam / p_ham)


all_counts = model["word_counts"]["spam"] + model["word_counts"]["ham"]
frequent = [w for w, c in all_counts.items() if c >= 2]
ratios = {w: word_log_ratio(model, w) for w in frequent}

print("most spammy:")
for word, r in heapq.nlargest(10, ratios.items(), key=lambda kv: kv[1]):
    print(f"  {word:12s} {r:+.2f}")

print("\nmost hammy:")
for word, r in heapq.nsmallest(10, ratios.items(), key=lambda kv: kv[1]):
    print(f"  {word:12s} {r:+.2f}")

most spammy:
  claim        +1.79
  click        +1.79
  cash         +1.38
  here         +1.38
  one          +1.38
  win          +1.38
  now          +1.10
  offer        +1.10
  only         +1.10
  waiting      +1.10

most hammy:
  i            -2.20
  can          -1.95
  the          -1.56
  great        -1.39
  meeting      -1.39
  we           -1.10
  review       -1.10
  thanks       -1.10
  m            -1.10
  late         -1.10


The spammy list is full of words you'd expect: "claim", "click", "cash", "win", "offer". The hammy list has personal, conversational words like "i", "we", "meeting" and "thanks". The model figured all of this out by counting.

Interestingly, "free" isn't in the top ten, even though it's in a lot of spam. That's because it also appears in several ham messages ("Are you free this weekend?"), so it's less of a giveaway than "claim", which only ever appears in spam.

Being able to inspect a model like this is a real advantage. If a deep learning model flags your boss's email as spam, it's hard to say why. With Naive Bayes you can point at the exact words.

## Step 10: Try your own messages

In [11]:
my_messages = [
    "You won a free trip, call now to claim it",
    "Can you call me after the meeting?",
    "Exclusive offer just for you, click the link",
    "The prize for the hackathon is a free GPU",
    "Are you free for dinner tonight?",
]

for msg in my_messages:
    probs = to_probabilities(log_scores(model, msg))
    label = "SPAM" if probs["spam"] > 0.5 else "ham "
    print(f"{label}  {probs['spam']:6.1%}  {msg}")

SPAM   96.5%  You won a free trip, call now to claim it
ham     0.1%  Can you call me after the meeting?
SPAM   89.4%  Exclusive offer just for you, click the link
ham     8.4%  The prize for the hackathon is a free GPU
SPAM   60.6%  Are you free for dinner tonight?


Look at the last one. "Are you free for dinner tonight?" is a perfectly normal message, but it got flagged. Here's why, word by word:

- "free" leans spam (it appeared 6 times in spam and 2 times in ham)
- "tonight" leans spam, because the only time the model ever saw it was in "Claim your free gift card now before it expires tonight"
- "are" and "for" lean slightly spam, just by chance
- only "dinner" leans ham

With 49 training messages, one example is enough to make "tonight" look like a spam word. This is the main weakness of small datasets: the model learns accidents of the data as if they were real patterns. With thousands of messages, "tonight" would show up in plenty of normal messages too, and the problem would go away.

Add your own messages to the list and see if you can fool it. Try a spam message without any of the obvious spam words, or a normal message that uses lots of them.

## Your turn

1. **Stop words.** Very common words like "the", "to", "you" and "a" carry little information. Make a `STOP_WORDS` set, filter them out in `tokenize`, and see if the results change. (Why should `STOP_WORDS` be a set and not a list?)

2. **Binary features.** Right now a word that appears three times in a message counts three times. Try counting each word at most once per message by using `set(tokenize(text))` in both training and prediction. This variant often works better for short texts.

3. **Bigrams.** Add pairs of consecutive words as extra tokens, so "credit card" becomes its own feature. Tuples are hashable, so `("credit", "card")` can go straight into a `Counter`. Use the `zip(words, words[1:])` trick from exercise 12.

4. **Adjust the threshold.** Instead of flagging anything above 50%, only flag messages above 90%. What happens to precision and recall? This is the precision/recall trade-off from Module 1 in action.

5. **Cross-check.** Change the seed in `random.Random(3)` to a few other values. How much does the accuracy move around? With a small test set, a single message changes the accuracy by several percent, which is a good reminder not to trust results from tiny test sets.

## What you used

- **`Counter`** for class counts and word counts, including adding two counters together
- **`defaultdict(Counter)`** for a separate word counter per class, created automatically
- **sets** for the vocabulary and fast membership checks, and for the test positions in the split
- **dicts** for the model, the log scores and the probabilities
- **`heapq.nlargest` / `nsmallest`** for the most spammy and hammy words
- **`dict.get` as a key function** with `max(scores, key=scores.get)`

And some ideas that come up everywhere in ML: priors, smoothing, working in log space to avoid underflow, softmax, the max-subtraction trick, precision vs recall, and inspecting what a model has learned.